# Practice 15 — pandas + NumPy
Work through each task in order. Try to solve it yourself before looking anything up!

In [1]:
import pandas as pd
import numpy as np

---
## Level 1 — Basics

### Task 1: DateTime
An employees DataFrame has been created for you.

1. Convert `hire_date` to datetime
2. Extract `hire_year` and `hire_month` as new columns
3. Add a `tenure_days` column = days between `hire_date` and `2026-01-01`
4. Find mean and median tenure using NumPy

In [91]:
# Starter data — don't change this
employees = pd.DataFrame({
    'name':      ['Ana', 'Ben', 'Cara', 'Dan', 'Eva', 'Finn', 'Gina', 'Hiro'],
    'dept':      ['Eng', 'Sales', 'Eng', 'HR', 'Sales', 'Eng', 'HR', 'Sales'],
    'hire_date': ['2019-03-15', '2021-07-01', '2020-11-22', '2018-06-30',
                  '2023-02-14', '2022-09-05', '2017-04-18', '2024-01-10'],
    'salary':    [95000, 72000, 88000, 61000, 78000, 102000, 67000, 84000],
})

# Your code here
employees['hire_date']= pd.to_datetime(employees['hire_date'])
employees['hire_year'] = employees['hire_date'].dt.year
employees['hire_month'] = employees['hire_date'].dt.month
employees['tenure_days'] = ((pd.to_datetime('2026-01-01')-employees['hire_date'])).dt.days
np.mean(employees['tenure_days'])
np.median(employees['tenure_days'])
#employees


np.float64(1755.5)

In [12]:
pd.to_datetime('2026-01-01')-employees['hire_date']

0   2484 days
1   1645 days
2   1866 days
3   2742 days
4   1052 days
5   1214 days
6   3180 days
7    722 days
Name: hire_date, dtype: timedelta64[ns]

### Task 2: MultiIndex
A product sales DataFrame has been created for you.

1. Set a MultiIndex from `category` and `region`, then sort it
2. Retrieve all rows for `'Electronics'`
3. Retrieve the single row for `('Furniture', 'West')`
4. Use `pd.IndexSlice` to get all `'East'` rows across every category
5. Find max revenue per category using NumPy

In [93]:
# Starter data — don't change this
np.random.seed(42)
categories = ['Electronics', 'Furniture', 'Clothing']
regions    = ['North', 'East', 'South', 'West']
products = pd.DataFrame({
    'category': categories * 4,
    'region':   [r for r in regions for _ in categories],
    'revenue':  np.random.randint(10_000, 100_000, size=12),
    'units':    np.random.randint(50, 500, size=12),
})

# Your code here
products = products.set_index(['category','region']).sort_index()
products.loc['Electronics']
products.loc[('Furniture','West')]
idx = pd.IndexSlice
products.loc[idx[:,'East'],:]
products.groupby(level='category')['revenue'].max()
for cat in products.index.get_level_values('category').unique():
    print(cat,np.max(products.loc[cat,'revenue']))

Clothing 92386
Electronics 70263
Furniture 97498


---
## Level 2 — Transformations

### Task 3: stack() / unstack() + Custom Logic
A wide DataFrame of monthly energy consumption by building has been created for you
(buildings as the index, months as columns).

1. Use `.stack()` to reshape into long format — each row one `(building, month)` pair. Store as `energy_long`.
2. Add a `log_kwh` column to `energy_long` using NumPy
3. Add a `high_use` column: `True` if consumption > 400 (use `np.where`)
4. From `energy_long`, `.unstack()` to restore wide format
5. Find the building with highest mean consumption using NumPy

In [96]:
# Starter data — don't change this
np.random.seed(17)
energy = pd.DataFrame({
    'building': ['Alpha', 'Beta', 'Gamma', 'Delta'],
    'Jan': np.random.randint(200, 600, size=4),
    'Mar': np.random.randint(200, 600, size=4),
    'Jun': np.random.randint(200, 600, size=4),
    'Sep': np.random.randint(200, 600, size=4),
    'Dec': np.random.randint(200, 600, size=4),
}).set_index('building')

# Your code here
energy_long = energy.stack()
energy_long = energy_long.to_frame(name = 'kwh')
energy_long['log_kwh'] = np.log(energy_long['kwh'])
energy_long['high_use'] = energy_long['kwh']>400 ##how to use np.where?
energy_long['high_use'] = np.where(energy_long['kwh']>400,True,False)
energy_wide =energy_long.unstack()
energy.mean(axis=1).idxmax()
#or
m = energy_long.groupby(level='building')['kwh'].mean()
m.index[np.argmax(m)]


'Alpha'

### Task 4: groupby + transform
A sales DataFrame has been created for you.

1. Add a `dept_avg_revenue` column = each row's department mean revenue (use `transform`)
2. Add a `above_dept_avg` column: `True` if `revenue` exceeds `dept_avg_revenue`
3. Bin `revenue` into 4 equal-width groups labelled `['Low', 'Mid', 'High', 'Top']`, store as `rev_tier`
4. Count rows per `rev_tier` (use `observed=True`)

In [58]:
# Starter data — don't change this
np.random.seed(33)
sales = pd.DataFrame({
    'rep':     ['Alice', 'Bob', 'Carol', 'Dan', 'Eve', 'Frank', 'Grace', 'Hank', 'Ida', 'Jake'],
    'dept':    ['Eng', 'Sales', 'Sales', 'Eng', 'HR', 'Sales', 'HR', 'Eng', 'Sales', 'HR'],
    'revenue': np.random.randint(20_000, 150_000, size=10),
    'deals':   np.random.randint(5, 50, size=10),
})

# Your code here
sales['dept_avg_revenue'] = sales.groupby('dept')['revenue'].transform("mean")
sales['above_dept_avg'] = sales['revenue']>sales['dept_avg_revenue']
sales['rev_tier'] = pd.cut(sales['revenue'], 4, labels=['Low','Mid','High','Top'])
sales.groupby('rev_tier',observed=True).agg('count')




,rep,dept,revenue,deals,dept_avg_revenue,above_dept_avg
rev_tier,,,,,,
Low,3,3,3,3,3,3
Mid,2,2,2,2,2,2
High,3,3,3,3,3,3
Top,2,2,2,2,2,2


---
## Level 3 — Aggregation

### Task 5: pd.merge() ✨ New

`pd.merge()` joins two DataFrames on a shared key column — similar to SQL JOIN.

```python
pd.merge(left, right, on='key')              # inner join (default) — only matching rows
pd.merge(left, right, on='key', how='left')  # left join — all left rows, NaN where no match
pd.merge(left, right, on='key', how='outer') # outer join — all rows from both
```

If the key column has different names in each DataFrame:
```python
pd.merge(left, right, left_on='id', right_on='customer_id')
```

Two DataFrames have been created for you: `customers` and `orders`.

1. Inner join `customers` and `orders` on `customer_id`. How many rows result?
2. Left join instead — how many rows now? Why is it different?
3. After the inner join, find mean `order_value` per `city` using groupby
4. Find which customer has the highest total order value — use groupby + NumPy to identify them

In [ ]:
# Starter data — don't change this
customers = pd.DataFrame({
    'customer_id': ['C001', 'C002', 'C003', 'C004', 'C005', 'C006'],
    'name':        ['Alice', 'Bob', 'Carol', 'Dan', 'Eve', 'Frank'],
    'city':        ['London', 'Paris', 'London', 'Berlin', 'Paris', 'Berlin'],
})

orders = pd.DataFrame({
    'order_id':    ['O1', 'O2', 'O3', 'O4', 'O5', 'O6', 'O7', 'O8'],
    'customer_id': ['C001', 'C002', 'C001', 'C003', 'C005', 'C002', 'C007', 'C003'],
    'order_value': [250, 180, 420, 90, 310, 75, 500, 640],
    'month':       ['Jan', 'Jan', 'Feb', 'Feb', 'Jan', 'Mar', 'Mar', 'Mar'],
})

# Your code here
im = pd.merge(
    customers,
    orders,
    how='inner',
    on='customer_id'
)
im.shape
lm = pd.merge(
    customers,
    orders,
    how='left',
    on='customer_id'
)
lm.shape
## 2 ids in customers but not in orders

im.groupby('city')['order_value'].mean()

s = im.groupby('customer_id')['order_value'].sum()
s.index[np.argmax(s)]
s.idxmax()

'C003'

### Task 6: Rolling Windows + .xs()
A monthly revenue DataFrame with a `(product, region)` MultiIndex has been created for you.

1. Use `.xs()` to pull out just the `'North'` region across all products
2. On that North slice, compute a 3-period rolling mean of `revenue`
3. Back on the full DataFrame, use `.xs()` to get the `'Gadget'` product across all regions
4. Find the month with the largest single-period revenue for `'Gadget'` in any region — use NumPy

In [99]:
# Starter data — don't change this
np.random.seed(61)
prods   = ['Gadget', 'Widget', 'Doohickey']
regions = ['North', 'South', 'East']
monthly = pd.DataFrame({
    'product': prods * 6,
    'region':  [r for r in regions * 2 for _ in prods],
    'month':   [m for m in ['Jan','Feb','Mar','Apr','May','Jun'] for _ in prods],
    'revenue': np.random.randint(5_000, 50_000, size=18),
}).set_index(['product', 'region'])

# Your code here
monthly
n = monthly.xs('North',level='region')
# how to do step 2?
n['revenue'].rolling(3).mean()
g = monthly.xs('Gadget',level='product')
g['month'].to_numpy()[np.argmax(g['revenue'])]
#or
g['month'].iloc[np.argmax(g['revenue'])]

'Jan'

In [98]:
n['revenue'].rolling(3).mean()

product
Gadget                NaN
Widget                NaN
Doohickey    31721.333333
Gadget       27457.666667
Widget       35490.000000
Doohickey    34564.666667
Name: revenue, dtype: float64

---
## Level 4 — Real-world

### Task 7: Full Pipeline
Two DataFrames have been created for you: `students` (demographics) and `scores` (test results).

1. Inner join `students` and `scores` on `student_id`
2. Drop any rows with null values
3. Add a `weighted_score` column = `(math * 0.4) + (english * 0.3) + (science * 0.3)`
4. Bin `weighted_score` into 3 groups labelled `['Pass', 'Merit', 'Distinction']`, store as `result`
5. Compute mean `weighted_score` grouped by `school` and `year`, then `.unstack()` so schools are rows and years are columns
6. Find the correlation between `math` and `science` scores using NumPy

In [90]:
# Starter data — don't change this
np.random.seed(88)
students = pd.DataFrame({
    'student_id': [f'S{i:03d}' for i in range(1, 21)],
    'name':       [f'Student_{i}' for i in range(1, 21)],
    'school':     np.random.choice(['Oakwood', 'Riverside', 'Hillcrest'], size=20),
    'year':       np.random.choice([10, 11, 12], size=20),
})

# scores has 18 rows — 2 students missing (no scores recorded)
scores = pd.DataFrame({
    'student_id': [f'S{i:03d}' for i in range(1, 19)],
    'math':       np.random.randint(40, 100, size=18),
    'english':    np.random.randint(40, 100, size=18),
    'science':    np.random.randint(40, 100, size=18),
})

# Your code here
ij = pd.merge(
    students,
    scores,
    how='inner',
    on='student_id'
)
ij = ij.dropna()
ij['weighted_score'] = np.dot(ij[['math','english','science']],[0.4,0.3,0.3])
ij['result'] = pd.cut(ij['weighted_score'], 3, labels= ['Pass','Merit','Distinction'])
ij.groupby(['school','year'])['weighted_score'].mean().unstack()
np.corrcoef(ij['math'],ij['science'])

array([[ 1.        , -0.40249304],
       [-0.40249304,  1.        ]])